# 04: LoRA Fine-tune (H3 Stretch Goal)

Trains a LoRA adapter for LLaVA-1.6-7B on the synthetic instruction-image
corpus. Configured for **A100 (40 GB)** — full `max_seq_len=1024` and the
default batch sizes. If Pro+ assigns you a T4 instead, reduce `max_seq_len`
to 768 and enable `gradient_checkpointing` (commented hint in the train cell).

**Estimated wallclock:** ~20-30 min on A100 (3 epochs, 200 examples).
**Estimated compute units:** ~6-7 on A100.

## Pro+ tip

Enable **'Run after disconnecting'** (Tools → Settings) and you can fire
this off, close the laptop, and come back to a finished adapter.

In [16]:
# Mount Google Drive. The first run prompts for OAuth; subsequent runs
# in the same session reuse the token automatically.
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
ROOT = pathlib.Path('/content/drive/MyDrive/grounded_vla')
ROOT.mkdir(parents=True, exist_ok=True)
print('drive root:', ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
drive root: /content/drive/MyDrive/grounded_vla


In [17]:
# Verify accelerator. Pro+ should give you A100 most of the time; if you
# see T4, change Runtime -> Change runtime type -> A100 GPU and re-run.
import subprocess
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                '--format=csv'])

CompletedProcess(args=['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv'], returncode=0)

In [18]:
# Clone repo if absent, then always pull to get the latest fixes.
# With an editable install (-e) a pull is all that's needed — no reinstall.
REPO_URL = 'https://github.com/KarthikRed2000/grounded_vla.git'
import os, subprocess
if not os.path.exists('/content/grounded_vla'):
    subprocess.run(['git', 'clone', REPO_URL, '/content/grounded_vla'], check=True)
subprocess.run(['git', '-C', '/content/grounded_vla', 'pull'], check=True)
%cd /content/grounded_vla

/content/grounded_vla


In [19]:
# Install repo + GPU stack. Quiet to keep the cell output sane.
!pip install -q -e .[gpu,data]

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for grounded_vla (pyproject.toml) ... done


In [20]:
import os
WEIGHTS = '/content/drive/MyDrive/grounded_vla/hf_cache'
DATA    = '/content/drive/MyDrive/grounded_vla/data'
os.environ['HF_HOME'] = WEIGHTS
os.environ['TRANSFORMERS_CACHE'] = WEIGHTS
os.environ['TRANSFORMERS_OFFLINE'] = '1'

In [21]:
import subprocess
result = subprocess.run(
    ['git', 'pull'],
    cwd='/content/grounded_vla',
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

Already up to date.




In [30]:
# Re-run this patched version (accepts 3 OR 4 args)
import json, torch
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset as TorchDataset

def _collate_fn(features):
    batch = {}
    max_len = max(f["input_ids"].shape[-1] for f in features)
    for key in features[0]:
        if key in ("pixel_values", "image_sizes"):
            batch[key] = torch.stack([f[key] for f in features])
        elif key in ("input_ids", "attention_mask", "labels"):
            pad_val = -100 if key == "labels" else 0
            tensors = []
            for f in features:
                t = f[key]
                gap = max_len - t.shape[-1]
                if gap > 0:
                    t = torch.cat([t, torch.full((*t.shape[:-1], gap), pad_val, dtype=t.dtype)], dim=-1)
                tensors.append(t)
            batch[key] = torch.stack(tensors)
        else:
            try: batch[key] = torch.stack([f[key] for f in features])
            except: pass
    return batch

def _patched_dataset(jsonl_path, images_dir, processor, max_seq_len=None):  # ← accepts 4 args
    rows = [json.loads(l) for l in Path(jsonl_path).read_text().splitlines() if l.strip()]
    images_dir = Path(images_dir)

    class _DS(TorchDataset):
        def __len__(self): return len(rows)
        def __getitem__(self, i):
            r = rows[i]
            img_path = Path(r["image_path"])
            if not img_path.is_absolute():
                img_path = images_dir / img_path
            image = Image.open(img_path).convert("RGB")
            gold = r["gold_actions"][0]
            target = ("Thought: Looking at the image, I will perform the grounded action.\n"
                      "Action: " + json.dumps(gold))
            chat = [
                {"role": "user",      "content": [{"type": "image"}, {"type": "text", "text": r["instruction"]}]},
                {"role": "assistant", "content": [{"type": "text", "text": target}]},
            ]
            enc = processor(images=image,
                            text=processor.apply_chat_template(chat, add_generation_prompt=False),
                            return_tensors="pt")
            item = {k: v.squeeze(0) for k, v in enc.items()}
            item["labels"] = item["input_ids"].clone()
            return item
    return _DS()

import grounded_vla.lora as _m
_m._load_synthetic_dataset = _patched_dataset
_m._collate_fn = _collate_fn
print("✓ hot-patched")

✓ hot-patched


In [31]:
# Train. The synthetic_sample fixture is tiny (3 examples) and only useful
# as a pipeline check; for real H3 numbers, swap to your real synthetic.jsonl
# (~200 reviewed examples). Edit JSONL/IMGS below.
from grounded_vla.lora import train_lora, LoRAConfig
from pathlib import Path

JSONL = f'{DATA}/synthetic_sample/synthetic_sample.jsonl'
IMGS  = f'{DATA}/synthetic_sample'
OUT   = Path('/content/drive/MyDrive/grounded_vla/checkpoints/llava-lora-r1')
OUT.mkdir(parents=True, exist_ok=True)

cfg = LoRAConfig(
    base_model=f'{WEIGHTS}/llava-v1.6-mistral-7b-hf',
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    learning_rate=2e-4,
    per_device_batch_size=1,
    gradient_accumulation_steps=8,
    num_epochs=3,
    max_seq_len=1024,  # A100 can handle full length; drop to 768 on T4
)
# T4 fallback: uncomment if Pro+ gave you a T4 instead of A100
# cfg.max_seq_len = 768
# from grounded_vla.lora import LoRAConfig as _C  # (gradient_checkpointing hook is in lora.py if needed)

train_lora(jsonl_path=JSONL, images_dir=IMGS, output_dir=OUT, config=cfg)

Loading weights:   0%|          | 0/687 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 15,990,784 || all params: 7,582,738,432 || trainable%: 0.2109


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,9.760432
20,5.107462
30,4.194388
40,4.062211
50,4.018895
60,4.000722
70,3.985886


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


[05/05/26 19:01:42] INFO     INFO:grounded_vla.lora:LoRA adapter saved to                                          
                             /content/drive/MyDrive/grounded_vla/checkpoints/llava-lora-r1

PosixPath('/content/drive/MyDrive/grounded_vla/checkpoints/llava-lora-r1')

In [32]:
# Verify adapter saved.
import os
for f in sorted(os.listdir('/content/drive/MyDrive/grounded_vla/checkpoints/llava-lora-r1')):
    print(f)

README.md
adapter_config.json
adapter_model.safetensors
checkpoint-25
checkpoint-50
checkpoint-75


## Done

Adapter is on Drive. Move on to `05_eval_with_lora.ipynb` for the H3 ablation.